In [ ]:

# Install required libraries

!pip install -q -U langchain langchain-community langchain-text-splitters langchain-google-genai faiss-cpu pypdf


In [ ]:

import os

# Google Colab file upload and secrets
from google.colab import files, userdata

# Document loader
from langchain_community.document_loaders import PyPDFLoader

# Text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Gemini embeddings and LLM
from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

# Vector database
from langchain_community.vectorstores import FAISS

# Prompt template
from langchain_core.prompts import ChatPromptTemplate

# Output parser
from langchain_core.output_parsers import StrOutputParser

print("All libraries imported successfully!")



/tmp/ipykernel_813/307965678.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


All libraries imported successfully!


In [ ]:
# Get API key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")

print("Gemini API connected successfully!")



Gemini API connected successfully!


In [ ]:
# Upload PDF file
uploaded_files = files.upload()

# Get PDF filename
pdf_path = list(uploaded_files.keys())[0]

print("Uploaded PDF:", pdf_path)



In [ ]:
# Initialize PDF loader
loader = PyPDFLoader(pdf_path)

# Extract text from PDF
documents = loader.load()

print("Total pages:", len(documents))

# Display first page
print("\nFirst page content:")
print(documents[0].page_content[:1000])


In [ ]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# Split documents into chunks
chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

# Display first chunk
print("\nFirst chunk:")
print(chunks[0].page_content)
print("\nSecond chunk:")
print(chunks[1].page_content)
print("\nThird chunk:")
print(chunks[2].page_content)


In [ ]:
# Initialize Gemini embedding model
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

print("Embedding model initialized successfully!")


In [ ]:
# Convert chunks into embeddings
# Store embeddings in FAISS vector database

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("FAISS vector database created successfully!")


In [ ]:
# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("Retriever created successfully!")


In [ ]:

from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize Gemini with a different model
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
    max_retries=3
)

print("Gemini LLM initialized successfully!")



In [ ]:
# Create prompt template

prompt = ChatPromptTemplate.from_template("""
You are an intelligent document analysis assistant.

Answer the user's question using ONLY the
information available in the provided context.

If the answer is not available in the context,
respond with:
"The information is not available in the uploaded document."

DOCUMENT CONTEXT:
{context}

USER QUESTION:
{question}

INSTRUCTIONS:
1. Understand the user's question.
2. Identify relevant information from the context.
3. Combine related information when necessary.
4. Give a clear and simple answer.
5. Do not invent information.
6. Use bullet points when appropriate.

ANSWER:
""")

print("Prompt template created successfully!")


In [ ]:
# Connect prompt + Gemini LLM + output parser

rag_chain = prompt | llm | StrOutputParser()

print("RAG chain created successfully!")


In [ ]:
def ask_question(question):

    # Step 1: Retrieve relevant document chunks
    retrieved_docs = retriever.invoke(question)

    # Step 2: Construct context
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    # Step 3: Combine context and question
    # Step 4: Send to Gemini and generate answer
    answer = rag_chain.invoke({
        "context": context,
        "question": question
    })

    # Step 5: Display answer
    print("\n" + "=" * 60)
    print("USER QUESTION")
    print("=" * 60)
    print(question)

    print("\n" + "=" * 60)
    print("GENERATED ANSWER")
    print("=" * 60)
    print(answer)

    # Step 6: Display source pages
    print("\n" + "=" * 60)
    print("SOURCE DOCUMENT PAGES")
    print("=" * 60)

    source_pages = sorted(set(
        doc.metadata.get("page", -1) + 1
        for doc in retrieved_docs
        if isinstance(doc.metadata.get("page"), int)
    ))

    print("Pages:", source_pages)

    return answer


print("Question answering function is ready!")


In [ ]:
# Get question from user
question = input("Ask your question: ")

# Generate answer using RAG
answer = ask_question(question)


In [ ]:
while True:

    question = input("\nAsk a question (type 'exit' to stop): ")

    if question.lower() == "exit":
        print("Thank you for using RAG Document Analyzer!")
        break

    ask_question(question)
